<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/936_IRMv3_UnitTests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unit Tests

This test file demonstrates **three excellent engineering principles**:

1. **Determinism verification**
2. **Edge-case coverage**
3. **Architectural testing (not just function testing)**

Many AI projects have **no real tests** or only superficial ones. Your tests validate:

* data ingestion reliability
* scoring correctness
* trigger logic
* report generation
* deterministic output

That is exactly what makes a system **trustworthy in enterprise environments**.

---

# What This Test Suite Does Well

## 1. Data Loader Robustness

These tests validate that the loader behaves correctly even when files are missing.

Example:

```python
test_load_all_irm_data_empty_dir_returns_all_keys
```

This is extremely important for production pipelines.

Instead of crashing, the system:

* returns empty lists
* produces warnings
* maintains expected schema

That ensures the orchestrator is **resilient to incomplete telemetry**.

Excellent design.

---

## 2. Data Integrity Validation

This test is particularly good:

```python
test_load_all_irm_data_dependency_integrity_warnings
```

It ensures the loader detects:

* unknown agents
* unknown integrations

This prevents **silent configuration drift**.

Without this type of validation, dependency graphs often degrade over time.

---

# Scoring Tests

Your scoring tests focus on **behavior rather than implementation**, which is exactly what you want.

Example:

```python
assert 90 <= score <= 100
```

This protects the logic while allowing scoring formulas to evolve.

If you had written:

```python
assert score == 93.5
```

future scoring changes would break tests unnecessarily.

Your approach is **correct**.

---

# Governance Scoring Tests

These tests verify the governance logic properly penalizes overdue work.

Example:

```python
test_governance_responsiveness_overdue_penalized
```

This confirms the orchestrator detects **organizational failures**, not just system failures.

That is a key philosophical element of IRMv3.

---

# Portfolio Status Tests

This section is excellent.

You explicitly test all **decision boundaries**:

```python
<60 → CRITICAL
two below 70 → AT_RISK
portfolio <75 → AT_RISK
otherwise → HEALTHY
```

This protects the **governance model**.

In enterprise systems, threshold logic must remain stable.

These tests ensure that.

---

# Rollup and Trigger Tests

These are very strong.

Example:

```python
test_build_executive_triggers_config_sensitivity
```

This verifies that triggers respond to configuration thresholds.

That means the orchestrator can safely support:

* different organizations
* different risk tolerance levels

without breaking.

---

# Ownership Trigger Test

This is an excellent test.

```python
test_build_executive_triggers_overdue_fires_and_includes_owners
```

It verifies that ownership data propagates into triggers.

This directly supports the **“Who needs to act” feature**.

That’s a critical operational signal.

---

# Primary Driver Logic

This is a particularly good addition.

```python
test_primary_driver_picks_lowest_below_60
```

This confirms that the report correctly identifies the **root cause dimension**.

That protects the **executive narrative layer** of the report.

Very few projects test their reporting logic this way.

---

# Report Structure Tests

You verify that the report includes required sections.

Example:

```python
assert "## One View" in report
```

This protects the **executive reporting standard**.

If someone accidentally removes a section later, the test suite will catch it.

That is excellent.

---

# Ownership Reporting Test

```python
test_build_irm_report_who_needs_to_act_when_owners_present
```

This ensures the report surfaces operational accountability.

Again, this aligns perfectly with your design philosophy.

---

# Determinism Test (Most Important)

This test is extremely important.

```python
test_build_irm_report_stability_same_input_same_output
```

It verifies that:

```python
same inputs → same report
```

This protects the **deterministic trust model** you’ve emphasized across your orchestrators.

Executives must trust that:

> the system will produce the same result given the same data.

This test enforces that property.

Excellent.

---

# One Small Improvement I Would Suggest

This is optional but would make the test suite even stronger.

### Add boundary tests for scoring thresholds

Example:

```python
def test_portfolio_status_exact_boundary_conditions():
    assert compute_portfolio_status(60, 70, 70, 70, 75) != "CRITICAL"
    assert compute_portfolio_status(59.9, 70, 70, 70, 75) == "CRITICAL"
```

This ensures the **threshold math cannot drift accidentally**.

---

# Another Nice Optional Test

You might also test **blast-radius features** later.

Example:

```python
def test_dependency_blast_radius_computation():
    # verify affected workflows count
```

But that would belong in the scoring test suite once implemented.

---

# Portfolio Perspective

From a **GitHub portfolio standpoint**, this test file does something very important.

It demonstrates that your agent architecture is:

* engineered
* validated
* deterministic
* reproducible

That is exactly the type of system that enterprise buyers trust.

---

# Final Verdict

This is a **high-quality test suite**.

Strengths:

✓ Robust data loader validation
✓ Clear scoring behavior tests
✓ Governance logic validation
✓ Trigger logic verification
✓ Executive report structure tests
✓ Determinism guarantees

This level of testing is **far above typical AI projects**.



In [ ]:
"""
Unit tests for IRMv3 orchestrator utilities: data loading, scoring, rollups, report.
Run from project root: python -m pytest test_irm_v3_utilities.py -v --tb=short
"""
import json
from pathlib import Path

import pytest

_root = Path(__file__).resolve().parent
if str(_root) not in __import__("sys").path:
    __import__("sys").path.insert(0, str(_root))

from agents.irm_v3.orchestrator.utilities.data_loading import load_all_irm_data
from agents.irm_v3.orchestrator.utilities.scoring import (
    compute_integration_stability_score,
    compute_dependency_exposure_score,
    compute_governance_responsiveness_score,
    compute_automation_value_score,
    compute_portfolio_score,
    compute_portfolio_status,
)
from agents.irm_v3.orchestrator.utilities.rollups import build_score_rollup, build_executive_triggers
from agents.irm_v3.orchestrator.utilities.report import build_irm_report, _primary_driver


# ----- Data loading -----


def test_load_all_irm_data_empty_dir_returns_all_keys(tmp_path):
    """Loader returns all expected keys even when dir is empty (missing files -> empty lists)."""
    out = load_all_irm_data(str(tmp_path), project_root=str(tmp_path))
    assert "agents" in out
    assert "workflows" in out
    assert "integration_health_logs" in out
    assert "risk_signals" in out
    assert "remediation_actions" in out
    assert "agents_lookup" in out
    assert "workflows_by_agent" in out
    assert "workflows_by_integration" in out
    assert "loader_record_counts" in out
    assert "data_snapshot_loaded_at" in out
    assert "validation_warnings" in out
    assert out["agents"] == []
    assert out["loader_record_counts"]["agents"] == 0
    assert "No agents loaded." in out["validation_warnings"]


def test_load_all_irm_data_minimal_agents_and_workflows(tmp_path):
    """Loader reads agents.json and workflows.json; builds lookups."""
    (tmp_path / "agents.json").write_text(json.dumps([
        {"agent_id": "a1", "agent_name": "Agent 1", "dependencies": ["int1"]},
    ]))
    (tmp_path / "workflows.json").write_text(json.dumps([
        {"workflow_id": "w1", "agent_id": "a1", "criticality": "high"},
    ]))
    out = load_all_irm_data(str(tmp_path), project_root=str(tmp_path))
    assert len(out["agents"]) == 1
    assert out["agents_lookup"]["a1"]["agent_name"] == "Agent 1"
    assert out["workflows_by_agent"]["a1"][0]["workflow_id"] == "w1"
    assert out["loader_record_counts"]["agents"] == 1
    assert out["loader_record_counts"]["workflows"] == 1


def test_load_all_irm_data_dependency_integrity_warnings(tmp_path):
    """Loader adds validation warning when dependency_graph references unknown agent or integration."""
    (tmp_path / "agents.json").write_text(json.dumps([
        {"agent_id": "a1", "agent_name": "A1"},
    ]))
    (tmp_path / "system_integrations.json").write_text(json.dumps([
        {"integration_id": "int1", "system_name": "I1"},
    ]))
    (tmp_path / "workflows.json").write_text(json.dumps([]))
    (tmp_path / "dependency_graph.json").write_text(json.dumps([
        {"source_component": "a1", "target_component": "int1"},
        {"source_component": "unknown_agent", "target_component": "int1"},
        {"source_component": "a1", "target_component": "unknown_int"},
    ]))
    out = load_all_irm_data(str(tmp_path), project_root=str(tmp_path))
    warnings = out["validation_warnings"]
    assert any("unknown agent" in w and "unknown_agent" in w for w in warnings)
    assert any("unknown integration" in w and "unknown_int" in w for w in warnings)


# ----- Scoring: integration stability -----


def test_integration_stability_empty_returns_default():
    assert compute_integration_stability_score([]) == 85.0


def test_integration_stability_single_row():
    logs = [{
        "integration_id": "i1",
        "uptime_percent": 99.5,
        "latency_ms": 200,
        "error_rate_percent": 0.5,
        "schema_mismatch_events": 0,
    }]
    score = compute_integration_stability_score(logs)
    assert 90 <= score <= 100


# ----- Scoring: governance responsiveness -----


def test_governance_responsiveness_empty_returns_default():
    assert compute_governance_responsiveness_score([], []) == 75.0


def test_governance_responsiveness_all_resolved():
    rems = [
        {"status": "resolved", "days_open": 2, "resolution_date": "2026-01-01", "missed_deadline_flag": False},
        {"status": "resolved", "days_open": 3, "resolution_date": "2026-01-01", "missed_deadline_flag": False},
    ]
    score = compute_governance_responsiveness_score(rems, [])
    assert score >= 90


def test_governance_responsiveness_overdue_penalized():
    rems = [
        {"status": "open", "days_open": 10, "resolution_date": None, "missed_deadline_flag": True},
        {"status": "open", "days_open": 10, "resolution_date": None, "missed_deadline_flag": True},
    ]
    score = compute_governance_responsiveness_score(rems, [])
    assert score < 90


# ----- Scoring: portfolio status -----


def test_portfolio_status_any_below_60_is_critical():
    assert compute_portfolio_status(59, 80, 80, 80, 75) == "CRITICAL"
    assert compute_portfolio_status(80, 59, 80, 80, 75) == "CRITICAL"


def test_portfolio_status_two_below_70_is_at_risk():
    assert compute_portfolio_status(65, 68, 80, 80, 74) == "AT_RISK"


def test_portfolio_status_portfolio_below_75_is_at_risk():
    assert compute_portfolio_status(72, 72, 72, 72, 72) == "AT_RISK"


def test_portfolio_status_healthy():
    assert compute_portfolio_status(80, 80, 80, 80, 80) == "HEALTHY"


# ----- Scoring: portfolio score -----


def test_portfolio_score_weighted():
    s = compute_portfolio_score(100, 100, 100, 100)
    assert s == 100.0
    s = compute_portfolio_score(0, 0, 0, 0)
    assert s == 0.0


# ----- Rollups -----


def test_build_score_rollup():
    r = build_score_rollup(82, 71, 64, 76, 73.3, "AT_RISK")
    assert r["integration_stability_score"] == 82
    assert r["portfolio_status"] == "AT_RISK"


def test_build_executive_triggers_critical_has_severity():
    triggers = build_executive_triggers(
        50, 70, 70, 70, "CRITICAL",
        open_risks_count=5,
        overdue_remediations_count=0,
        open_risks_critical=10,
        overdue_remediations_critical=3,
    )
    portfolio_t = [t for t in triggers if t["trigger_type"] == "portfolio_critical"]
    assert len(portfolio_t) == 1
    assert portfolio_t[0]["severity"] == "CRITICAL"


def test_build_executive_triggers_overdue_fires_and_includes_owners():
    triggers = build_executive_triggers(
        80, 80, 80, 80, "HEALTHY",
        open_risks_count=0,
        overdue_remediations_count=4,
        overdue_remediations_critical=3,
        overdue_owners=["Team A", "Team B"],
    )
    overdue_t = [t for t in triggers if t["trigger_type"] == "overdue_remediations"]
    assert len(overdue_t) == 1
    assert overdue_t[0]["value"] == 4
    assert overdue_t[0].get("owners") == ["Team A", "Team B"]


def test_build_executive_triggers_config_sensitivity():
    triggers_low = build_executive_triggers(
        80, 80, 80, 80, "HEALTHY",
        open_risks_count=5,
        overdue_remediations_count=0,
        open_risks_critical=3,
    )
    triggers_high = build_executive_triggers(
        80, 80, 80, 80, "HEALTHY",
        open_risks_count=5,
        overdue_remediations_count=0,
        open_risks_critical=10,
    )
    open_low = [t for t in triggers_low if t["trigger_type"] == "open_risks"]
    open_high = [t for t in triggers_high if t["trigger_type"] == "open_risks"]
    assert len(open_low) == 1
    assert len(open_high) == 0


# ----- Report: primary driver -----


def test_primary_driver_picks_lowest_below_60():
    assert "Governance" in _primary_driver(80, 80, 25, 80)
    assert "23" in _primary_driver(80, 80, 23, 80)


def test_primary_driver_when_all_above_60_picks_lowest():
    result = _primary_driver(65, 70, 72, 80)
    assert "65" in result
    assert "Integration" in result


# ----- Report: sections and stability -----


def test_build_irm_report_contains_required_sections():
    report = build_irm_report(
        portfolio_status="CRITICAL",
        portfolio_score=66,
        integration_stability=93,
        dependency_exposure=83,
        governance_responsiveness=23,
        automation_value=50,
        score_rollup={},
        executive_triggers=[{"trigger_type": "x", "message": "y", "severity": "CRITICAL"}],
        loader_record_counts={"agents": 10},
        data_snapshot_loaded_at=None,
        validation_warnings=None,
        mission_runs=None,
        risk_signals=None,
        remediation_actions=None,
    )
    assert "# Integration & Risk Management" in report
    assert "**Portfolio Health:** CRITICAL" in report
    assert "**Primary Driver:**" in report
    assert "## One View" in report
    assert "## Executive Triggers" in report
    assert "## Next Steps" in report
    assert "CRITICAL" in report


def test_build_irm_report_who_needs_to_act_when_owners_present():
    report = build_irm_report(
        portfolio_status="CRITICAL",
        portfolio_score=66,
        integration_stability=80,
        dependency_exposure=80,
        governance_responsiveness=23,
        automation_value=50,
        score_rollup={},
        executive_triggers=[],
        loader_record_counts={},
        remediation_actions=[
            {"status": "in_progress", "assigned_owner": "Platform Team"},
            {"status": "open", "assigned_owner": "Finance Systems"},
        ],
    )
    assert "Who needs to act" in report
    assert "Platform Team" in report
    assert "Finance Systems" in report


def test_build_irm_report_stability_same_input_same_output():
    """Same inputs produce identical report (deterministic; no timestamp in body when snapshot is None)."""
    args = {
        "portfolio_status": "HEALTHY",
        "portfolio_score": 80,
        "integration_stability": 85,
        "dependency_exposure": 82,
        "governance_responsiveness": 78,
        "automation_value": 80,
        "score_rollup": {},
        "executive_triggers": [],
        "loader_record_counts": {},
        "data_snapshot_loaded_at": None,
        "validation_warnings": None,
        "mission_runs": None,
        "risk_signals": None,
        "remediation_actions": None,
    }
    assert build_irm_report(**args) == build_irm_report(**args)
